In [1]:
import pandas as pd

df = pd.read_csv('data/thai_wikipron_5-4-2026.csv')
df

,writing,phonetic,count
0,ก,kɔː˧,2
1,ก,kɔː˧.kaj˨˩,2
2,ก.,kɔː˧,1
3,ก.ค.,kɔː˧.kʰɔː˧,1
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1
...,...,...,...
18305,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧,2
18306,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˥˩,2
18307,ไฮ้,haj˦˥,1
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1


In [2]:
import re

def normalize_phonetic(phonetic: str) -> str:
    phonetic = (
        phonetic
        .replace('p̚', 'p')
        .replace('t̚', 't')
        .replace('k̚', 'k')
        .replace('a̯', 'ə')
        .replace('˥˩', '˦˩')
        .replace('˩˩˦', '˨˥')
    )

    SHORT_VOWELS = ['a', 'i', 'ɯ', 'u', 'e', 'ɤ', 'o', 'ɛ', 'ɔ']

    pattern = (
        '(' + '|'.join(map(re.escape, SHORT_VOWELS)) + ')'
        r'([˥˦˧˨˩]+)(?=\.|$)'
    )

    phonetic = re.sub(pattern, r'\1ʔ\2', phonetic)

    phonetic = re.sub(r'[.…]+', '.', phonetic)

    phonetic = re.sub(r'^\.|\.$', '', phonetic)

    return phonetic

print(normalize_phonetic('….daj˧.….nɯŋ˨˩'))

daj˧.nɯŋ˨˩


In [3]:
df['normalized_phonetic'] = df['phonetic'].apply(normalize_phonetic)
df

,writing,phonetic,count,normalized_phonetic
0,ก,kɔː˧,2,kɔː˧
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩
2,ก.,kɔː˧,1,kɔː˧
3,ก.ค.,kɔː˧.kʰɔː˧,1,kɔː˧.kʰɔː˧
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1,kɔː˧.tʰɔː˧.mɔː˧
...,...,...,...,...
18305,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧,2,haj˧.droː˧.t͡ɕeːn˧
18306,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˥˩,2,haj˧.droː˧.t͡ɕen˦˩
18307,ไฮ้,haj˦˥,1,haj˦˥
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,daj˧.nɯŋ˨˩


In [4]:
import thai_gpa

def evaluate(text: str, ipa: str) -> tuple:
    result = thai_gpa.align(text, ipa)
    reconstructed_text = ''.join(s.reconstruct_text() for s in result)
    reconstructed_ipa = '.'.join(s.get_ipa(is_reduplicated=s.is_reduplicated) for s in result)
    return reconstructed_text, reconstructed_ipa

print(evaluate('การขัดกันของผลประโยชน์', 'kaːn˧.kʰat˨˩.kan˧.kʰɔːŋ˨˥.pʰon˨˥.praʔ˨˩.joːt˨˩'))

('การขัดกันของผลประโยชน์', 'kaːn˧.kʰat˨˩.kan˧.kʰɔːŋ˨˥.pʰon˨˥.praʔ˨˩.joːt˨˩')


In [5]:
from tqdm.auto import tqdm
tqdm.pandas()

def apply_evaluate(row):
    try:
        reconstructed_text, phonetic_answer = evaluate(row['writing'], row['normalized_phonetic'])
    except Exception as e:
        reconstructed_text, phonetic_answer = None, f'ERROR: {e}'
    return pd.Series([reconstructed_text, phonetic_answer])

df[['reconstructed_text', 'phonetic_answer']] = df.progress_apply(apply_evaluate, axis=1)
df.to_csv('data/test.csv', index=False)
df

C:\Users\pawi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
 91%|█████████ | 16675/18310 [04:09<00:11, 137.94it/s]c:\Storage\repos\thai_grapheme_sandbox\thai_ipa.py:109: UserWarning: Warning: No explicit glottal stop at "e"
  warnings.warn(f'Warning: No explicit glottal stop at "{original}"')
100%|██████████| 18310/18310 [04:23<00:00, 69.37it/s] 


,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
0,ก,kɔː˧,2,kɔː˧,None,ERROR: Could not align 'ก' with 'kɔː˧'
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩,None,ERROR: Could not align 'ก' with 'kɔː˧.kaj˨˩'
2,ก.,kɔː˧,1,kɔː˧,None,ERROR: Could not align 'ก.' with 'kɔː˧'
3,ก.ค.,kɔː˧.kʰɔː˧,1,kɔː˧.kʰɔː˧,None,ERROR: Could not align 'ก.ค.' with 'kɔː˧.kʰɔː˧'
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1,kɔː˧.tʰɔː˧.mɔː˧,None,ERROR: Could not align 'ก.ท.ม.' with 'kɔː˧.tʰɔ...
...,...,...,...,...,...,...
18305,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧,2,haj˧.droː˧.t͡ɕeːn˧,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧
18306,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˥˩,2,haj˧.droː˧.t͡ɕen˦˩,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˦˩
18307,ไฮ้,haj˦˥,1,haj˦˥,ไฮ้,haj˦˥
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,daj˧.nɯŋ˨˩,None,ERROR: Could not align '…ใด…หนึ่ง' with 'daj˧....


In [6]:
mask = ~df["phonetic_answer"].str.startswith("ERROR:")
mismatches = df.loc[
    mask & (df["normalized_phonetic"] != df["phonetic_answer"]),
    ["writing", "normalized_phonetic", "phonetic_answer"]
]

mismatches

,writing,normalized_phonetic,phonetic_answer


In [7]:
errors = df[df["phonetic_answer"].str.startswith("ERROR:")]
errors

,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
0,ก,kɔː˧,2,kɔː˧,None,ERROR: Could not align 'ก' with 'kɔː˧'
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩,None,ERROR: Could not align 'ก' with 'kɔː˧.kaj˨˩'
2,ก.,kɔː˧,1,kɔː˧,None,ERROR: Could not align 'ก.' with 'kɔː˧'
3,ก.ค.,kɔː˧.kʰɔː˧,1,kɔː˧.kʰɔː˧,None,ERROR: Could not align 'ก.ค.' with 'kɔː˧.kʰɔː˧'
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1,kɔː˧.tʰɔː˧.mɔː˧,None,ERROR: Could not align 'ก.ท.ม.' with 'kɔː˧.tʰɔ...
...,...,...,...,...,...,...
18262,ไอซ์แลนด์,ʔajs˦˥.lɛːn˧,1,ʔajs˦˥.lɛːn˧,None,ERROR: argument of type 'NoneType' is not iter...
18263,ไอดอล,ʔaj˧.dɔl˥˩,2,ʔaj˧.dɔl˦˩,None,ERROR: argument of type 'NoneType' is not iter...
18278,ไอศครีม,ʔajs˧.kʰriːm˧,2,ʔajs˧.kʰriːm˧,None,ERROR: argument of type 'NoneType' is not iter...
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,daj˧.nɯŋ˨˩,None,ERROR: Could not align '…ใด…หนึ่ง' with 'daj˧....


In [8]:
errors.sample(10)   

,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
11179,ส,sɔː˩˩˦.sɯa̯˩˩˦,2,sɔː˨˥.sɯə˨˥,None,ERROR: Could not align 'ส' with 'sɔː˨˥.sɯə˨˥'
6576,บัณฑิตย,ban˧.dit̚˨˩.ta˨˩.ja˦˥.,2,ban˧.dit˨˩.taʔ˨˩.jaʔ˦˥,None,ERROR: Could not align 'บัณฑิตย' with 'ban˧.di...
7429,ปาเลสไตน์,paː˧.leːs˦˥.taːj˧,1,paː˧.leːs˦˥.taːj˧,None,ERROR: argument of type 'NoneType' is not iter...
5715,ธรรมาภิบาล,tʰam˧.maː˧.pʰi˦˥.baːn˧,1,tʰam˧.maː˧.pʰiʔ˦˥.baːn˧,None,ERROR: Could not align 'ธรรมาภิบาล' with 'tʰam...
7351,ปัญญาอ่อน,pan˧.jaː˧.ʔɔːn˨˩,1,pan˧.jaː˧.ʔɔːn˨˩,None,ERROR: Could not align 'ปัญญาอ่อน' with 'pan˧....
8947,มอสส์,mɔːs˦˥,2,mɔːs˦˥,None,ERROR: argument of type 'NoneType' is not iter...
16231,เอนทัลปี,ʔeːn˧.tʰaw˧.piː˥˩,2,ʔeːn˧.tʰaw˧.piː˦˩,None,ERROR: Could not align 'เอนทัลปี' with 'ʔeːn˧....
14148,ฯพณฯ,pʰa˦˥.na˦˥.tʰan˥˩,1,pʰaʔ˦˥.naʔ˦˥.tʰan˦˩,None,ERROR: Could not align 'ฯพณฯ' with 'pʰaʔ˦˥.naʔ...
5011,ถถถ,tʰɔː˩˩˦.tʰɔː˩˩˦.tʰɔː˩˩˦,2,tʰɔː˨˥.tʰɔː˨˥.tʰɔː˨˥,None,ERROR: Could not align 'ถถถ' with 'tʰɔː˨˥.tʰɔː...
6826,บ่มิไก๊,bɔː˨˩.mi˦˥.kaj˦˥,1,bɔː˨˩.miʔ˦˥.kaj˦˥,None,ERROR: Could not align 'บ่มิไก๊' with 'bɔː˨˩.m...


In [9]:
mask = ~errors["normalized_phonetic"].str.contains(
    r"[lsf][˥˦˧˨˩]", regex=True, na=False
)
filtered_errors = errors[mask]
filtered_errors

,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
0,ก,kɔː˧,2,kɔː˧,None,ERROR: Could not align 'ก' with 'kɔː˧'
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩,None,ERROR: Could not align 'ก' with 'kɔː˧.kaj˨˩'
2,ก.,kɔː˧,1,kɔː˧,None,ERROR: Could not align 'ก.' with 'kɔː˧'
3,ก.ค.,kɔː˧.kʰɔː˧,1,kɔː˧.kʰɔː˧,None,ERROR: Could not align 'ก.ค.' with 'kɔː˧.kʰɔː˧'
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1,kɔː˧.tʰɔː˧.mɔː˧,None,ERROR: Could not align 'ก.ท.ม.' with 'kɔː˧.tʰɔ...
...,...,...,...,...,...,...
18041,ไปรษณียบัตร,praj˧.sa˨˩.niː˧.bat̚˨˩,2,praj˧.saʔ˨˩.niː˧.bat˨˩,None,ERROR: Could not align 'ไปรษณียบัตร' with 'pra...
18044,ไปรษณีย์อิเล็กทรอนิกส์,praj˧.sa˨˩.niː˧.ʔi˨˩.lek̚˦˥.tʰrɔː˧.nik̚˨˩,1,praj˧.saʔ˨˩.niː˧.ʔiʔ˨˩.lek˦˥.tʰrɔː˧.nik˨˩,None,ERROR: Could not align 'ไปรษณีย์อิเล็กทรอนิกส์...
18132,ไม้กอล์ฟ,maːj˦˥.kɔːp̚˦˥,2,maːj˦˥.kɔːp˦˥,None,ERROR: Could not align 'ไม้กอล์ฟ' with 'maːj˦˥...
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,daj˧.nɯŋ˨˩,None,ERROR: Could not align '…ใด…หนึ่ง' with 'daj˧....


In [12]:
filtered_errors.sample(10)

,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
8113,พฤ.,pʰa˦˥.rɯʔ˦˥,2,pʰaʔ˦˥.rɯʔ˦˥,None,ERROR: Could not align 'พฤ.' with 'pʰaʔ˦˥.rɯʔ˦˥'
1745,ข,kʰɔː˩˩˦.kʰaj˨˩,2,kʰɔː˨˥.kʰaj˨˩,None,ERROR: Could not align 'ข' with 'kʰɔː˨˥.kʰaj˨˩'
6575,บัณฑิต,ban˧.dit̚˨˩,1,ban˧.dit˨˩,None,ERROR: Could not align 'บัณฑิต' with 'ban˧.dit˨˩'
14071,อ๋อง,ʔɔŋ˩˩˦,1,ʔɔŋ˨˥,None,ERROR: Could not align 'อ๋อง' with 'ʔɔŋ˨˥'
5948,นักษัตรบดี,nak̚˦˥.sat̚˨˩.tra˨˩.bɔː˧.diː˧,1,nak˦˥.sat˨˩.traʔ˨˩.bɔː˧.diː˧,None,ERROR: Could not align 'นักษัตรบดี' with 'nak˦...
13406,ออริกาโน,ʔɔː˧.ri˦˥.kaː˧.noː˥˩,1,ʔɔː˧.riʔ˦˥.kaː˧.noː˦˩,None,ERROR: Could not align 'ออริกาโน' with 'ʔɔː˧.r...
6403,บดินทร,bɔː˧.din˧.tʰra˦˥.,1,bɔː˧.din˧.tʰraʔ˦˥,None,ERROR: Could not align 'บดินทร' with 'bɔː˧.din...
11185,ส/ค,sɔː˩˩˦.kʰɔː˧,1,sɔː˨˥.kʰɔː˧,None,ERROR: Could not align 'ส/ค' with 'sɔː˨˥.kʰɔː˧'
16535,แซลมอน,sɛw˧.mɔn˥˩,2,sɛw˧.mɔn˦˩,None,ERROR: Could not align 'แซลมอน' with 'sɛw˧.mɔn˦˩'
1938,ขาอ่อน,kʰaː˩˩˦.ʔɔːn˨˩,1,kʰaː˨˥.ʔɔːn˨˩,None,ERROR: Could not align 'ขาอ่อน' with 'kʰaː˨˥.ʔ...
